# Train the probabilistic LSTM in Google Colab

This notebook trains the repository's custom probabilistic LSTM on the airline-passenger time series. It is a focused Colab version of `passengers_LSTM_ProbAct.ipynb`; the Python practice cells at the end of that notebook are left out here.

## Before you run it

1. Push the local `development` branch to GitHub once (instructions are in `README.md`). Colab can only clone branches that exist on GitHub.
2. Open this notebook from the `development` branch in Colab.
3. In Colab, choose **Runtime → Change runtime type → GPU** if a GPU is available to your account.
4. Run all cells. The setup cell clones the model and data files, then asks you to connect Google Drive so checkpoints survive runtime resets.

The free Colab tier may not offer a GPU every time. The notebook will report the device it received and can run on CPU as a fallback, though training may be much slower.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/gabrielmadirolas/probabilistic-networks.git"
REPOSITORY_BRANCH = "development"
REPOSITORY_DIR = Path("/content/probabilistic-networks")

# When this notebook is opened from GitHub, its code runs in /content rather than
# inside the repository. Clone only the source and CSV; the repository also has
# large historical checkpoints that this training run does not need.
if Path("models/custom_lstms.py").is_file() and Path("data/airline_passengers.csv").is_file():
    PROJECT_DIR = Path.cwd()
else:
    if not (REPOSITORY_DIR / ".git").exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPOSITORY_DIR),
            ],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "sparse-checkout", "set", "models", "data"],
        check=True,
    )
    PROJECT_DIR = REPOSITORY_DIR

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    CHECKPOINT_ROOT = Path("/content/drive/MyDrive/probabilistic-networks/colab_runs")
else:
    CHECKPOINT_ROOT = PROJECT_DIR / "checkpoints_colab"

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_DIR}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")

In [ ]:
'''
Trains an LSTM model, using probabilistic activation functions as described in this paper:
https://paperswithcode.com/paper/probact-a-probabilistic-activation-function

To use such custom activation function, a custom LSTM cell has to be implemented.
For this, I will use code modified from this file:
https://github.com/pytorch/pytorch/blob/main/benchmarks/fastrnns/custom_lstms.py

I store both implementations in the "models" folder.

The task and the original code which I have modified for this notebook comes from the following tutorial:
https://machinelearningmastery.com/lstm-for-time-series-prediction-in-pytorch/
where the task is defined like this:
"The problem you will look at in this post is the international airline passengers prediction problem.
This is a problem where, given a year and a month, the task is to predict the number of international airline passengers in units of 1,000."
The data ranges from January 1949 to December 1960, or 12 years, with 144 observations.

The data is downloade from here:
https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv

''';

### Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
import os
import sys
import importlib
import glob
import re

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

from models.probact import EWTrainableMuSigma

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Training device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Define general variables 

In [ ]:
#proportion of data used for training
prop_train = 2/3

### Fetch data 

In [ ]:
data_folder = './data/'
data_file = 'airline_passengers.csv'

numpass_type = 'float32' # to convert the number of passengers from int to float

df = pd.read_csv(data_folder + data_file)
time_series = df[["Passengers"]].values.astype(numpass_type)

# see how data looks like and whether it makes sense
plt.plot(time_series)
plt.show()

In [ ]:
# train-test split the time series
train_size = int(len(time_series) * prop_train)
test_size = len(time_series) - train_size
train, test = time_series[:train_size], time_series[train_size:]

In [ ]:
# create a function to generate a dataset of fixed window from a time series
lookback = 4 # number of time steps to be fed to the model
def create_dataset(dataset, lookback):
    '''
    Transforms a time series into a prediction dataset

    Args:
        dataset: A numpy array of time series, first dimension is the time steps
        lookback: Size of window for prediction
    '''

    num_features = train.shape[1]
    num_samples = len(dataset)-lookback
    X, y = np.empty((num_samples,lookback,num_features), dtype=numpass_type), np.empty((num_samples,lookback,num_features), dtype=numpass_type)
    
    for i in range(len(dataset)-lookback):
        feature = dataset[i:i+lookback]
        target = dataset[i+1:i+lookback+1] # Should the target include the full window? Does it make any difference?
        #target = dataset[i+lookback] # this does not include the full window in the target
        #y[i] = [target] # this does not include the full window in the target
        X[i] = feature # this includes the full window in the target
        y[i] = target # this includes the full window in the target

    return torch.tensor(X), torch.tensor(y)

In [ ]:
# create_dataset is designed to convert a time series into a tensor of dimensions: 
# (number of samples, time steps, features)
# num_samples = number of sub-series; time steps = length of sub-serie
# ergo: num_samples = len(data set) - lookback; time_steps = lookback

X_train, y_train = create_dataset(train, lookback=lookback)
X_test, y_test = create_dataset(test, lookback=lookback)
''' 
# Some numbers and examples to understand dimensionalities:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(train.ndim)
print(len(train),len(train[0]))
print(X_train[:5])
''';

## Probabilistic LSTM 

### Import the custom LSTM and probabilistic activation 

In [ ]:
from models import custom_lstms, probact

### Define variables 

In [ ]:
# Model and training settings
num_features = X_train.shape[2]
hidden_size = 500
n_LSTM = 1
dropout = 0.0
dropout_linear = 0.0
batch_size = 32
batch_first = True
bidirectional = False
proj_size = 0
lr = 0.001

# Keep RUN_NAME unchanged to resume this run. Choose a new name to start another.
RUN_NAME = "airline_probact_run_01"
TOTAL_EPOCHS = 10000
RECORD_STEP = 500
RESUME_TRAINING = True
USE_TORCH_COMPILE = False  # Optional; ordinary PyTorch is the safer Colab default.

probabilistic_sharing = "per_node"  # per_node, per_layer, or per_network
noise_distribution = "normal"  # normal or variance-matched uniform
noise_location = "pre_activation"  # or post_activation
sample_mode = "train_only"  # always, train_only, or never

prob_params = {
    "mean_mu": 0.0,
    "std_mu": 0.0,
    "mean_sigma": 0.25,
    "std_sigma": 0.05,
    "alpha": 0.0,
    "beta": 0.0,
}
train_mu = bool(prob_params["std_mu"])  # Set explicitly to override.
train_sigma = True

### Define model

In [ ]:
# Build the LSTM model to predict the time series. With lookback=1, accuracy won't be great
# The model is created as a class, with a LSTM layer and a fully-connected layer


class AirModel(nn.Module):
    def __init__(self):
        super().__init__()
        lstm_output_size = (proj_size or hidden_size) * (2 if bidirectional else 1)
        self.lstm = custom_lstms.prob_lstm(
            num_features=num_features, hidden_size=hidden_size,
            num_layers=n_LSTM, dropout=dropout, batch_first=batch_first,
            prob_params=prob_params, sharing=probabilistic_sharing,
            distribution=noise_distribution, noise_location=noise_location,
            sample_mode=sample_mode, train_mu=train_mu, train_sigma=train_sigma,
            bidirectional=bidirectional, proj_size=proj_size,
        )
        self.dropout = nn.Dropout(p=dropout_linear)
        self.linear = nn.Linear(lstm_output_size, 1)
        #self.probact = EWTrainableMuSigma(hidden_size, prob_params)
    def forward(self, x):
        x, states = self.lstm(x) 
        x = self.dropout(x)
        x = self.linear(x)
        #x = self.probact(x)
        return x  # output layout follows batch_first
'''The output of prob_lstm() is a tuple. The first element is the generated hidden states [from the last LSTM layer],
one for each time step of the input. The second element contains the final hidden and cell states for all layers and directions; it is not used here. 
''';

In [ ]:
test_model = AirModel()
print(test_model.lstm)
print(test_model.linear)

### Train the LSTM network

In [ ]:
# Train with MSE loss. Checkpoints are written periodically so a disconnected
# Colab runtime can resume from Google Drive on the next session.

checkpoint_dir = CHECKPOINT_ROOT / RUN_NAME
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_state_path = checkpoint_dir / "latest_training_state.pt"

if not RESUME_TRAINING and (
    latest_state_path.exists()
    or list(checkpoint_dir.glob("saved_model_epoch*.pth"))
):
    raise ValueError(
        "This run already has checkpoints. Keep RESUME_TRAINING=True to continue, "
        "or choose a new RUN_NAME to start a separate run."
    )

X_train, y_train, X_test, y_test = [
    tensor.to(device) for tensor in (X_train, y_train, X_test, y_test)
]
base_model = AirModel().to(device)
optimizer = optim.Adam(base_model.parameters(), lr=lr)
loss_fn = nn.MSELoss()
loader = data.DataLoader(
    data.TensorDataset(X_train, y_train), shuffle=True, batch_size=batch_size
)


def clean_state_dict(state):
    # Older checkpoints were saved from torch.compile() and prefix keys with
    # _orig_mod. Strip that prefix so they also load into an ordinary model.
    return {
        key.removeprefix("_orig_mod."): value
        for key, value in state.items()
    }


def checkpoint_epoch(path):
    return int(path.stem.removeprefix("saved_model_epoch"))


start_epoch = 0
if RESUME_TRAINING and latest_state_path.exists():
    saved = torch.load(latest_state_path, map_location=device)
    model_state = clean_state_dict(saved["model_state_dict"])
    legacy_scale_parameterization = not any(
        key.endswith("scale_parameterization_version") for key in model_state
    )
    base_model.load_state_dict(model_state)
    if legacy_scale_parameterization:
        print("Loaded legacy model weights; optimizer state was reset for the new positive-scale parameterization")
    else:
        optimizer.load_state_dict(saved["optimizer_state_dict"])
    start_epoch = int(saved["epoch"])
    print(f"Resuming from epoch {start_epoch}")
elif RESUME_TRAINING:
    weight_files = list(checkpoint_dir.glob("saved_model_epoch*.pth"))
    if weight_files:
        last_weight = max(weight_files, key=checkpoint_epoch)
        base_model.load_state_dict(
            clean_state_dict(torch.load(last_weight, map_location=device))
        )
        start_epoch = checkpoint_epoch(last_weight)
        print(f"Resuming model weights from epoch {start_epoch}; optimizer starts fresh")
    else:
        print("No checkpoint found; starting a new run")
else:
    print("RESUME_TRAINING is False; starting from fresh model weights")

model = base_model
def predict_batch_first(inputs):
    model_input = inputs if batch_first else inputs.transpose(0, 1)
    predictions = model(model_input)
    return predictions if batch_first else predictions.transpose(0, 1)

if USE_TORCH_COMPILE:
    if hasattr(torch, "compile"):
        model = torch.compile(base_model)
    else:
        print("torch.compile is unavailable in this runtime; continuing without it")

print(f"Training device: {device}")
print(f"Saving checkpoints in: {checkpoint_dir}")

start_time = time.time()
for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = predict_batch_first(X_batch)
        # batch_first=True returns [batch, sequence, features]
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    completed_epoch = epoch + 1
    if completed_epoch % RECORD_STEP != 0 and completed_epoch != TOTAL_EPOCHS:
        continue

    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
    model_state = model_to_save.state_dict()
    weights_path = checkpoint_dir / f"saved_model_epoch{completed_epoch}.pth"
    temporary_weights_path = weights_path.with_suffix(".tmp")
    torch.save(model_state, temporary_weights_path)
    temporary_weights_path.replace(weights_path)

    training_state = {
        "epoch": completed_epoch,
        "model_state_dict": model_state,
        "optimizer_state_dict": optimizer.state_dict(),
    }
    temporary_state_path = latest_state_path.with_suffix(".tmp")
    torch.save(training_state, temporary_state_path)
    temporary_state_path.replace(latest_state_path)

    model.eval()
    with torch.no_grad():
        train_pred = predict_batch_first(X_train)
        train_rmse = torch.sqrt(loss_fn(train_pred, y_train)).item()
        test_pred = predict_batch_first(X_test)
        test_rmse = torch.sqrt(loss_fn(test_pred, y_test)).item()
    print(
        f"Epoch {completed_epoch}: train RMSE {train_rmse:.4f}, "
        f"test RMSE {test_rmse:.4f}"
    )

print(f"Training finished in {time.time() - start_time:.1f} seconds")

### Verify the network

In [ ]:
'''
It is expected to see the RMSE of test set is about an order of magnitude larger. 
The RMSE of 100 means the prediction and the actual target would be
in average off by 100 in value (i.e., 100,000 passengers in this dataset).
To better understand the prediction quality, let's plot the output using matplotlib
'''
with torch.no_grad():
    # shift train predictions for plotting
    train_plot = np.ones_like(time_series) * np.nan
    train_plot[lookback:train_size] = predict_batch_first(X_train)[:, -1, 0].cpu()
    # shift test predictions for plotting
    test_plot = np.ones_like(time_series) * np.nan
    test_plot[train_size+lookback:len(time_series)] = predict_batch_first(X_test)[:, -1, 0].cpu()
# plot
plt.plot(time_series)
plt.plot(train_plot)
plt.plot(test_plot)
plt.show()